In [ ]:
import sys
import os
import pandas as pd
import polars as pl

current_dir = os.getcwd()
src_dir = os.path.dirname(current_dir)
sys.path.append(src_dir)

from data_preprocessing.data_normalization import BMLLNormalizer
from data_preprocessing.data_access import DataAccessFactory
from data_preprocessing.repartition import Repartition
from data_preprocessing.reconciliation import Reconciliation
from feature_engineering.order_flow import OrderFlowFeatureEngineering

from pipeline.config import PipelineConfig, DataConfig, ProcessingConfig, StorageConfig, RayConfig, S3Location
from pipeline.pipeline import Pipeline

# Shared configuration
BAR_DURATION_MS = 250

In [ ]:
# Test Parallel File Discovery
import time
import ray

# Initialize Ray if not already
if not ray.is_initialized():
    ray.init()

# Test parallel discovery
data_access = DataAccessFactory.create('s3', region='us-east-1', profile_name='blitvinfdp')
base_path = 's3://orderflowanalysis/intermediate/repartitioned_v2'
threshold = 100

# Step 1: Sequential prefix discovery
bucket = base_path.replace('s3://', '').split('/')[0]
base_prefix = '/'.join(base_path.replace('s3://', '').split('/')[1:]) + '/'

print('Step 1: Sequential prefix discovery...')
prefixes = data_access._discover_prefixes_sequential(bucket, base_prefix, threshold)
print(f'Found {len(prefixes)} prefixes for parallel processing')

# Step 2: Parallel file listing
print('\nStep 2: Parallel file listing...')
start_time = time.time()

files = data_access._list_files_parallel(prefixes)

end_time = time.time()

print(f'✅ Parallel discovery completed in {end_time - start_time:.2f} seconds')
print(f'Found {len(files)} files total')
print(f'Sample files:')
for i, (path, size) in enumerate(files[:5]):
    print(f'  {i+1}. {path} ({size:.3f} GB)')